# Projeto 9 — Calculadora e Avaliador de Expressões

**Disciplina:** Inteligência Artificial / IA LLM Web Services — PUC-SP  
**Objetivo:** receber uma pergunta em linguagem natural, deixar um modelo de linguagem escolher uma ferramenta e delegar o cálculo exato ao Python.

**Sem chave da OpenAI:** o exemplo principal usa um modelo Qwen local pelo Ollama, por meio da biblioteca `openai` e sua interface compatível. O valor `api_key="ollama"` é apenas um campo exigido pelo cliente; o servidor local o ignora. Não há chamada à API da OpenAI. A opção Hugging Face, ao fim, funciona no Colab com um token `HF_TOKEN`/`HF_KEY` e pode consumir créditos da conta.

**Fluxo:** pergunta → modelo escolhe `calculate` ou `solve_equation` → Python valida e executa → resultado JSON → modelo explica a resposta.

> Não use `eval`/`exec` sobre texto produzido por usuário ou modelo. O parser abaixo aceita somente uma pequena lista de operações matemáticas. As chamadas de ferramenta são validadas antes da execução.


## 1. Preparação

Execute as células na ordem. O cálculo e os testes locais funcionam sem conta nem chave. Para testar as respostas em linguagem natural, instale o [Ollama](https://ollama.com/download) no computador onde o Jupyter está rodando e execute no terminal:

```bash
ollama pull qwen3:1.7b
ollama serve
```

O aplicativo Ollama pode já estar servindo na porta 11434; nesse caso, `ollama serve` não é necessário. O modelo `qwen3:1.7b` ocupa cerca de 1,4 GB para download. O modelo menor `qwen3:0.6b` não devolveu a chamada de ferramenta exigida em nosso teste local.

**Google Colab:** o endereço `localhost` do Colab se refere à máquina remota do Colab, não ao seu PC. Para usar o notebook no Colab, escolha a opção Hugging Face na seção 7 e adicione o token aos Secrets. Nenhuma chave da OpenAI é necessária.


In [ ]:
%pip -q install "openai>=1.0" "sympy>=1.12"


In [1]:
import ast
import json
import os
from typing import Any

import sympy as sp
from openai import OpenAI

print("SymPy:", sp.__version__)


SymPy: 1.14.0


## 2. Ferramenta matemática segura

O programa aceita números, parênteses, `+ - * / **` (ou `^`), as constantes `pi` e `e`, e as funções `sqrt`, `abs`, `sin`, `cos` e `log`. Para equações, aceita uma variável (`x`) e polinômios de grau até 4. Tamanho de entrada, número de nós, expoentes e magnitude são limitados.


In [2]:
MAX_INPUT = 160
MAX_NODES = 60
MAX_EXPONENT = 12
MAX_ABS_NUMBER = 1_000_000

ALLOWED_FUNCTIONS = {
    "sqrt": sp.sqrt,
    "abs": sp.Abs,
    "sin": sp.sin,
    "cos": sp.cos,
    "log": sp.log,
}
ALLOWED_CONSTANTS = {"pi": sp.pi, "e": sp.E}

def parse_math(source: str, variable: str | None = None) -> sp.Expr:
    if not isinstance(source, str) or len(source) > MAX_INPUT:
        raise ValueError("Expressão ausente ou longa demais.")
    source = source.replace("^", "**")
    tree = ast.parse(source, mode="eval")
    if sum(1 for _ in ast.walk(tree)) > MAX_NODES:
        raise ValueError("Expressão complexa demais.")

    symbol = sp.Symbol(variable, real=True) if variable else None

    def convert(node: ast.AST) -> sp.Expr:
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            if abs(node.value) > MAX_ABS_NUMBER:
                raise ValueError("Número fora do limite permitido.")
            return sp.Rational(str(node.value))
        if isinstance(node, ast.Name):
            if variable and node.id == variable:
                return symbol
            if node.id in ALLOWED_CONSTANTS:
                return ALLOWED_CONSTANTS[node.id]
            raise ValueError(f"Nome não permitido: {node.id}")
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = convert(node.operand)
            return value if isinstance(node.op, ast.UAdd) else -value
        if isinstance(node, ast.BinOp):
            left, right = convert(node.left), convert(node.right)
            if isinstance(node.op, ast.Add): return left + right
            if isinstance(node.op, ast.Sub): return left - right
            if isinstance(node.op, ast.Mult): return left * right
            if isinstance(node.op, ast.Div):
                if right == 0: raise ZeroDivisionError("Divisão por zero.")
                return left / right
            if isinstance(node.op, ast.Pow):
                if not right.is_Integer or abs(int(right)) > MAX_EXPONENT:
                    raise ValueError("Expoente deve ser inteiro entre -12 e 12.")
                exponent = int(right)
                if left == 0 and exponent < 0:
                    raise ZeroDivisionError("Divisão por zero.")
                if symbol is not None and left.has(symbol):
                    degree = sp.degree(left, symbol)
                    if exponent < 0 or degree * exponent > 4:
                        raise ValueError("Esta versão resolve polinômios de grau até 4.")
                if left.is_Rational:
                    bits = max(left.p.bit_length(), left.q.bit_length())
                    if bits * abs(exponent) > 4096:
                        raise ValueError("Resultado grande demais.")
                return left ** right
            raise ValueError("Operador não permitido.")
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id not in ALLOWED_FUNCTIONS or node.keywords or len(node.args) != 1:
                raise ValueError("Função não permitida ou argumentos inválidos.")
            return ALLOWED_FUNCTIONS[node.func.id](convert(node.args[0]))
        raise ValueError("Sintaxe não permitida.")

    result = convert(tree.body)
    if not result.is_finite and result.is_finite is not None:
        raise ValueError("Resultado não finito.")
    return result

def calculate(expression: str) -> dict[str, Any]:
    try:
        result = parse_math(expression)
        if result.free_symbols:
            raise ValueError("Use solve_equation para variáveis.")
        approximation = sp.N(result, 12)
        if not approximation.is_finite:
            raise ValueError("Resultado não finito.")
        return {"ok": True, "expression": expression, "exact": str(result),
                "decimal": str(approximation)}
    except (ValueError, SyntaxError, ZeroDivisionError, TypeError, OverflowError) as exc:
        return {"ok": False, "error": str(exc)}

def solve_equation(equation: str, variable: str = "x") -> dict[str, Any]:
    try:
        if variable != "x":
            raise ValueError("Esta versão aceita apenas a variável x.")
        if not isinstance(equation, str) or equation.count("=") != 1:
            raise ValueError("Escreva a equação com um único sinal de igual.")
        lhs_text, rhs_text = equation.split("=")
        lhs = parse_math(lhs_text.strip(), variable)
        rhs = parse_math(rhs_text.strip(), variable)
        x = sp.Symbol(variable, real=True)
        polynomial = sp.Poly(lhs - rhs, x)
        if polynomial.is_zero:
            return {"ok": True, "equation": equation, "variable": variable,
                    "solutions": "todos os números reais"}
        if polynomial.degree() > 4:
            raise ValueError("Esta versão resolve polinômios de grau até 4.")
        roots = sp.solve(polynomial.as_expr(), x)
        return {"ok": True, "equation": equation, "variable": variable,
                "solutions": [{"exact": str(root), "decimal": str(sp.N(root, 12))}
                              for root in roots]}
    except (ValueError, SyntaxError, ZeroDivisionError, TypeError,
            OverflowError, sp.PolynomialError) as exc:
        return {"ok": False, "error": str(exc)}


## 3. Testes do motor de cálculo

Estas verificações não usam modelo, rede nem chave. Elas demonstram resultado exato, solução de equação e bloqueio de código Python arbitrário.


In [3]:
assert calculate("(2 + 3) * 7")["exact"] == "35"
assert calculate("1/3 + 1/6")["exact"] == "1/2"
assert calculate("sqrt(2)")["ok"]
assert solve_equation("x^2 - 5*x + 6 = 0")["solutions"] == [
    {"exact": "2", "decimal": "2.00000000000"},
    {"exact": "3", "decimal": "3.00000000000"},
]
assert not calculate("__import__('os').system('echo unsafe')")["ok"]
assert not calculate("2**1000000")["ok"]
assert not calculate("(((2**12)**12)**12)**12")["ok"]
assert not solve_equation("x^5 = 1")["ok"]
assert solve_equation("x=x")["solutions"] == "todos os números reais"
print("9 verificações passaram.")
print(calculate("(17/3)^2 + sqrt(81)"))
print(solve_equation("x^2 - 5*x + 6 = 0"))


9 verificações passaram.
{'ok': True, 'expression': '(17/3)^2 + sqrt(81)', 'exact': '370/9', 'decimal': '41.1111111111'}
{'ok': True, 'equation': 'x^2 - 5*x + 6 = 0', 'variable': 'x', 'solutions': [{'exact': '2', 'decimal': '2.00000000000'}, {'exact': '3', 'decimal': '3.00000000000'}]}


## 4. Contrato das ferramentas

O modelo recebe apenas a descrição das funções e propõe argumentos JSON. O aplicativo decide se executa. A implementação nunca interpreta texto do modelo como código Python.


In [4]:
TOOLS = [
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Calculate a mathematical expression exactly. Use for arithmetic and functions, not equations.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "Math expression, e.g. (17/3)^2 + sqrt(81)"}},
            "required": ["expression"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "solve_equation",
        "description": "Solve a polynomial equation in x, degree at most 4, e.g. x^2 - 5*x + 6 = 0.",
        "parameters": {"type": "object", "properties": {
            "equation": {"type": "string", "description": "Equation with exactly one = sign."},
            "variable": {"type": "string", "description": "Variable name; must be x."}},
            "required": ["equation"], "additionalProperties": False}}},
]

SYSTEM = (
    "Você é um tutor de matemática. Para qualquer cálculo ou equação, chame a ferramenta "
    "apropriada. Nunca estime resultados numéricos por conta própria. Depois, explique em "
    "português, cite o resultado exato e, se útil, o decimal. Se a ferramenta retornar "
    "erro, explique o limite sem inventar um resultado."
)

def dispatch_tool(name: str, raw_arguments: str) -> dict[str, Any]:
    try:
        if not isinstance(raw_arguments, str) or len(raw_arguments) > 500:
            raise ValueError("Argumentos longos demais.")
        args = json.loads(raw_arguments)
        if not isinstance(args, dict):
            raise ValueError("Argumentos devem ser um objeto JSON.")
        if name == "calculate" and set(args) == {"expression"}:
            return calculate(args["expression"])
        if name == "solve_equation" and set(args) <= {"equation", "variable"} and "equation" in args:
            return solve_equation(args["equation"], args.get("variable", "x"))
        raise ValueError("Ferramenta ou argumentos não permitidos.")
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return {"ok": False, "error": str(exc)}

assert dispatch_tool("calculate", '{"expression":"2+2"}')["exact"] == "4"
assert not dispatch_tool("run_python", '{"code":"print(1)"}')["ok"]
print("Contrato das ferramentas validado.")


Contrato das ferramentas validado.


## 5. Agente: chamada, execução e resposta

A primeira resposta do modelo deve conter uma chamada de ferramenta. O Python registra a chamada e o resultado; a segunda resposta do modelo explica esse resultado. `tool_choice="required"` evita que o modelo responda à pergunta matemática sem usar a ferramenta.


In [5]:
def ask_calculator(question: str, client: OpenAI, model: str) -> dict[str, Any]:
    if not isinstance(question, str) or not question.strip() or len(question) > 500:
        raise ValueError("Pergunta vazia ou longa demais.")
    local_options = ({"reasoning_effort": "none"}
                     if str(getattr(client, "base_url", "")).startswith("http://localhost:11434/") else {})
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question.strip()},
    ]
    first = client.chat.completions.create(
        model=model, messages=messages, tools=TOOLS, tool_choice="required",
        temperature=0, max_tokens=256, **local_options
    )
    assistant = first.choices[0].message
    calls = assistant.tool_calls or []
    if not calls or len(calls) > 3:
        raise RuntimeError("O modelo não retornou entre 1 e 3 chamadas de ferramenta.")
    messages.append({
        "role": "assistant", "content": assistant.content,
        "tool_calls": [{"id": call.id, "type": "function", "function": {
            "name": call.function.name, "arguments": call.function.arguments}}
                       for call in calls],
    })
    trace = []
    for call in calls:
        result = dispatch_tool(call.function.name, call.function.arguments)
        try:
            logged_arguments = json.loads(call.function.arguments)
        except (ValueError, TypeError):
            logged_arguments = {"raw": str(call.function.arguments)[:200]}
        trace.append({"tool": call.function.name,
                      "arguments": logged_arguments, "result": result})
        messages.append({"role": "tool", "tool_call_id": call.id,
                         "content": json.dumps(result, ensure_ascii=False)})
    second = client.chat.completions.create(
        model=model, messages=messages, tool_choice="none", temperature=0,
        max_tokens=256, **local_options
    )
    return {"question": question, "answer": second.choices[0].message.content,
            "trace": trace}


## 6. Teste do fluxo sem rede

O cliente simulado emite uma chamada de ferramenta no mesmo formato da API. Assim, verificamos a integração e a trilha de execução sem fingir que um modelo real foi executado.


In [6]:
from types import SimpleNamespace as NS

class FakeCompletions:
    def __init__(self): self.requests = []
    def create(self, **kwargs):
        self.requests.append(kwargs)
        if len(self.requests) == 1:
            call = NS(id="demo_call_1", function=NS(
                name="solve_equation", arguments=json.dumps({"equation": "x^2-5*x+6=0"})))
            return NS(choices=[NS(message=NS(content=None, tool_calls=[call]))])
        tool_result = json.loads(kwargs["messages"][-1]["content"])
        assert [item["exact"] for item in tool_result["solutions"]] == ["2", "3"]
        return NS(choices=[NS(message=NS(content="As soluções são x = 2 e x = 3."))])

fake = NS(chat=NS(completions=FakeCompletions()))
demo = ask_calculator("Resolva x² - 5x + 6 = 0", fake, "modelo-simulado")
assert demo["trace"][0]["result"]["ok"]
assert fake.chat.completions.requests[0]["tool_choice"] == "required"
print(json.dumps(demo, indent=2, ensure_ascii=False))


{
  "question": "Resolva x² - 5x + 6 = 0",
  "answer": "As soluções são x = 2 e x = 3.",
  "trace": [
    {
      "tool": "solve_equation",
      "arguments": {
        "equation": "x^2-5*x+6=0"
      },
      "result": {
        "ok": true,
        "equation": "x^2-5*x+6=0",
        "variable": "x",
        "solutions": [
          {
            "exact": "2",
            "decimal": "2.00000000000"
          },
          {
            "exact": "3",
            "decimal": "3.00000000000"
          }
        ]
      }
    }
  ]
}


## 7. Execução com modelo real, sem chave da OpenAI

**Opção A — Jupyter no seu computador (Ollama):** execute as duas células abaixo após iniciar o Ollama e baixar `qwen3:0.6b`. O servidor local ignora o valor em `api_key`.

**Opção B — Google Colab (Hugging Face):** execute a célula de configuração da opção B em vez da opção A. No painel *Secrets*, adicione um token `HF_TOKEN` ou `HF_KEY` com permissão para Inference Providers. As chamadas podem consumir créditos; consulte a página do provedor antes de executar. O modelo padrão da célula B é um exemplo e pode precisar ser trocado por um modelo com ferramentas disponível na sua conta.


In [7]:
# Opção A: Jupyter local + Ollama (sem chave e sem serviço de inferência pago).
MODEL = "qwen3:1.7b"
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama", timeout=60)
print("Cliente configurado para Ollama local:", MODEL)


Cliente configurado para Ollama local: qwen3:1.7b


In [8]:
# Opção B: Google Colab + Hugging Face. Execute SOMENTE se escolher esta opção.
# Ela substitui o client/model da opção A; nunca imprima o token.
USE_HUGGING_FACE = False
if USE_HUGGING_FACE:
    try:
        from google.colab import userdata
        token = None
        for secret_name in ("HF_TOKEN", "HF_KEY"):
            try:
                token = userdata.get(secret_name)
            except Exception:
                pass
            if token:
                break
    except ImportError:
        token = os.getenv("HF_TOKEN") or os.getenv("HF_KEY")
    if not token:
        raise RuntimeError("Adicione HF_TOKEN ou HF_KEY aos Secrets do Colab.")
    client = OpenAI(base_url="https://router.huggingface.co/v1",
                    api_key=token, timeout=90)
    MODEL = "openai/gpt-oss-20b:fastest"
    print("Cliente configurado para Hugging Face:", MODEL)


In [9]:
# O Ollama local foi instalado e testado; no Colab, configure antes a opção B.
RUN_LIVE_DEMO = True
if RUN_LIVE_DEMO:
    for question in [
        "Quanto é 1/3 + 1/6?",
        "Resolva x^2 - 5*x + 6 = 0.",
    ]:
        try:
            result = ask_calculator(question, client, MODEL)
            print("PERGUNTA:", result["question"])
            print("RESPOSTA:", result["answer"])
            print("TRILHA:", json.dumps(result["trace"], ensure_ascii=False))
            print()
        except Exception as exc:
            print("Não foi possível completar a chamada:", type(exc).__name__, str(exc)[:300])
            break
else:
    print("Demo remota/local desativada. Testes sem rede concluídos nas seções 3 e 6.")


PERGUNTA: Quanto é 1/3 + 1/6?
RESPOSTA: A soma de $ \frac{1}{3} $ e $ \frac{1}{6} $ é:

**Exato:** $ \frac{1}{2} $

**Decimal:** $ 0.500000000000 $
TRILHA: [{"tool": "calculate", "arguments": {"expression": "(1/3) + (1/6)"}, "result": {"ok": true, "expression": "(1/3) + (1/6)", "exact": "1/2", "decimal": "0.500000000000"}}]

PERGUNTA: Resolva x^2 - 5*x + 6 = 0.
RESPOSTA: A equação $ x^2 - 5x + 6 = 0 $ pode ser resolvida encontrando as raízes usando a fórmula quadrática ou por fatoração.

### Passo 1: Fatoração
A equação pode ser fatorada como:
$$
(x - 2)(x - 3) = 0
$$

### Passo 2: Soluções
As soluções são:
$$
x = 2 \quad \text{e} \quad x = 3
$$

### Resultado Exato
As soluções exatas são:
$$
x = 2 \quad \text{e} \quad x = 3
$$

### Resultado Decimal
As soluções decimais são:
$$
x = 2.00000000000 \quad \text{e} \quad x = 3.00000000000
$$

Portanto, as soluções da equação são $ x = 2 $ e $ x = 3 $.
TRILHA: [{"tool": "solve_equation", "arguments": {"equation": "x^2 - 5*x + 6 = 0", "varia

## 8. Avaliação e limites

- **Correção:** os resultados matemáticos vêm do SymPy, não da previsão do LLM. Verifique a trilha `trace` em cada pergunta.
- **Segurança:** nomes desconhecidos, atributos, listas, importações, chamadas arbitrárias, expoentes grandes e expressões excessivas são rejeitados. O modelo só pode escolher entre duas funções validadas.
- **Cobertura:** esta versão resolve polinômios em `x` de grau até 4 e expressões com as operações indicadas. Não resolve inequações, derivadas, sistemas ou equações transcendentais.
- **Modelo pequeno:** `qwen3:1.7b` reduz o consumo de recursos, mas ainda pode produzir argumentos de ferramenta incorretos. O aplicativo devolve um erro controlado; um modelo maior pode ser mais confiável.
- **Reprodutibilidade:** as células 3 e 6 validam o Python sem rede. Os dois exemplos da seção 7 foram executados com Ollama local em 17/11/2026 e suas saídas foram salvas neste notebook. Em outro ambiente, configure Ollama ou Hugging Face antes de executá-los novamente.

**Fontes:** [OpenAI Docs — Chat Completions e tools](https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create) · [Ollama — compatibilidade com cliente OpenAI](https://docs.ollama.com/api/openai-compatibility) · [Ollama — tool calling](https://docs.ollama.com/capabilities/tool-calling) · [Qwen3 no Ollama](https://ollama.com/library/qwen3) · [Hugging Face — function calling](https://huggingface.co/docs/inference-providers/guides/function-calling)
